### 实验标题
**基于PyTorch的多层感知机（MLP）手写数字识别实验**
### 实验日期
2026年7月13日
### 实验目的
1. **掌握PyTorch深度学习框架的基本流程**  
   - 熟练使用`torch.utils.data.DataLoader`加载并批处理数据，理解训练/验证/测试集的划分方法。  
   - 学会定义自定义模型（此处为MLP），并运用`nn.Sequential`搭建包含全连接层、激活函数、Dropout和批归一化的网络结构。
2. **熟悉模型训练与评估的标准范式**  
   - 实现训练循环（前向传播、损失计算、反向传播、梯度裁剪、参数更新）。  
   - 实现验证与测试阶段的评估函数，输出损失、准确率及预测结果。  
   - 使用学习率调度器（`ReduceLROnPlateau`）动态调整学习率，提升训练稳定性。
3. **实践模型性能可视化与结果分析**  
   - 记录训练过程中的损失和准确率变化，并绘制曲线图，直观观察模型收敛情况。  
   - 通过保存最优模型权重，实现模型选择与最终测试性能评估。
4. **解决实际工程中的环境问题**  
5. **为后续更复杂模型（如CNN、RNN）的实验提供可复用的代码模板**，构建完整的深度学习实验闭环。

**数据集**：MNIST 手写数字数据集（10 类，0~9）

**样本数量**：
- 原始训练集：60,000 张
- 测试集：10,000 张
- 实际划分后：训练集约 54,000 张，验证集约 6,000 张（从训练集中随机抽取 10%）

**图像规格**：28×28 灰度图，展平为 784 维向量作为模型输入。

**预处理**：
- 转为张量并归一化到 [0,1]
- 标准化（均值 0.1307，标准差 0.3081）
- 展平为 784 维

**批处理**：`batch_size=128`，训练集打乱顺序，验证/测试集不打乱。

**特点**：经典基准数据集，类别均衡，计算轻量，适合快速验证算法性能。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import numpy as np
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt          # 新增：绘图库

# ---------- 全局辅助函数 ----------
def flatten_image(x):
    return x.view(-1)

### 数据加载与探索（EDA）

#### 1. 数据加载
本实验使用 `torchvision.datasets.MNIST` 自动下载 MNIST 数据集，并通过 `transforms.Compose` 组合三步预处理：
- **ToTensor()**：将 PIL 图像转为张量，像素值缩放到 [0,1]；
- **Normalize(mean=0.1307, std=0.3081)**：标准化，使数据分布接近标准正态分布；
- **Lambda(flatten_image)**：将 (1,28,28) 展平为 784 维向量，适配 MLP 输入。

#### 2. 数据集划分与加载器
- 原始训练集（60,000 张）按 9:1 随机划分为**训练集**（约 54,000 张）和**验证集**（约 6,000 张），测试集（10,000 张）保持不变，划分随机种子固定为 42 保证可复现。
- 使用 `DataLoader` 批处理：`batch_size=128`，训练集打乱顺序，验证/测试集不打乱，启用 2 个子进程并行加载。

#### 3. 探索性分析
- **类别分布**：训练集中每个数字约有 5,400 张，验证集约 600 张，测试集每类约 1,000 张，类别均衡。
- **数据形状**：每批数据张量形状为 `(128, 784)`，标签为 `(128,)` 的整数。
- **像素值**：原始值 0~255，标准化后均值接近 0、标准差接近 1，有助于梯度稳定。
- **样本可视化（可选）**：可通过 `matplotlib` 展示批次中的前几张图像及其标签，验证预处理效果。

以上步骤确保了数据加载正确、划分合理，为模型训练和评估提供了可靠的基础。

In [2]:
# ---------- 数据加载 ----------
def get_mnist_loaders(batch_size=128, val_split=0.1, num_workers=2):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
        transforms.Lambda(flatten_image)
    ])

    train_full = torchvision.datasets.MNIST(
        root='./data', train=True, download=True, transform=transform
    )
    test_set = torchvision.datasets.MNIST(
        root='./data', train=False, download=True, transform=transform
    )

    num_train = len(train_full)
    indices = list(range(num_train))
    split = int(np.floor(val_split * num_train))
    np.random.seed(42)
    np.random.shuffle(indices)
    train_indices, val_indices = indices[split:], indices[:split]

    train_set = Subset(train_full, train_indices)
    val_set = Subset(train_full, val_indices)

    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True
    )
    val_loader = DataLoader(
        val_set, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True
    )
    test_loader = DataLoader(
        test_set, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True
    )
    return train_loader, val_loader, test_loader

### 模型设计说明
#### 1. 模型类型
采用**多层感知机（Multilayer Perceptron, MLP）**，一种全连接前馈神经网络，用于手写数字图像的分类任务。MLP 结构简单、易于实现，适合作为基线模型验证数据加载和训练流程的正确性。
#### 2. 网络架构
模型通过 `torch.nn.Sequential` 堆叠多个线性层和非线性变换，具体结构如下（输入 → 输出）：
| 层序号 | 层类型 | 输入维度 | 输出维度 | 参数/激活 | 说明 |
|:---:|:---|:---:|:---:|:---|:---|
| 1 | `Linear` | 784 | 512 | 权重 + 偏置 | 输入层 → 第一个隐藏层 |
| 2 | `ReLU` | – | – | `ReLU()` | 引入非线性，缓解梯度消失 |
| 3 | `Dropout` | – | – | `p=0.3` | 随机丢弃 30% 神经元，防止过拟合 |
| 4 | `Linear` | 512 | 256 | 权重 + 偏置 | 隐藏层 → 第二个隐藏层 |
| 5 | `BatchNorm1d` | 256 | 256 | 可学习缩放/偏移 | 批归一化，加速收敛并稳定训练 |
| 6 | `ReLU` | – | – | `ReLU()` | 非线性激活 |
| 7 | `Linear` | 256 | 10 | 权重 + 偏置 | 输出层，输出 10 类 logits |

**注**：输出层未使用 Softmax，因为损失函数 `CrossEntropyLoss` 内部包含 LogSoftmax 和负对数似然，直接接受 logits。
#### 3. 参数规模
- 全连接层参数数 = `(输入维度 × 输出维度) + 输出维度`（偏置）
- 第一层：784×512 + 512 ≈ **402,432**
- 第二层：512×256 + 256 ≈ **131,328**
- 第三层：256×10 + 10 = **2,570**
- 批归一化层：每层有 2 个可学习参数（缩放和偏移），共 256×2 = **512**（此处仅一层 BN）
- **总可训练参数** ≈ **536,842**，属于轻量级模型。
#### 4. 设计思路与选择依据
- **隐藏层宽度**：选择 512 和 256 作为隐层维度，在计算开销和表达能力间取得平衡，足以建模 MNIST 的简单模式。
- **激活函数**：统一使用 **ReLU**，因其计算简单且能有效缓解梯度消失，相比 sigmoid/tanh 收敛更快。
- **正则化**：
  - **Dropout（概率 0.3）**：在第一个隐藏层后引入，随机丢弃部分神经元，降低特征间的协同适应，增强泛化能力。
  - **批归一化（BatchNorm）**：置于第二个隐藏层前，对每批数据归一化，允许使用更高学习率，并起到轻微正则化作用。
- **输出层**：无额外激活，直接输出 logits，配合 `CrossEntropyLoss` 实现多分类。
#### 5. 损失函数与优化
- **损失函数**：`CrossEntropyLoss`（适用于多分类任务，内部结合 Softmax 和 NLLLoss）。
- **优化器**：Adam，初始学习率 1e-3，自适应调整学习率，适合中小规模模型。
- **学习率调度**：`ReduceLROnPlateau`，监控验证损失，若连续 3 个 epoch 未改善则学习率减半（factor=0.5），帮助跳出局部最优。
#### 6. 梯度裁剪
在每个训练步骤后执行 `clip_grad_norm_(max_norm=1.0)`，防止梯度爆炸，保持训练稳定。
#### 7. 模型特点总结
- 结构清晰，易于修改和扩展；
- 融合了 Dropout 和批归一化两种正则化手段，兼顾表现力与泛化；
- 参数量可控，适合快速迭代；
- 输出 logits 的设计与现代分类损失函数兼容。

In [3]:
# 模型
mlp = nn.Sequential(
    nn.Linear(784, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Linear(256, 10)
)
model = mlp

### 训练设置说明

#### 1. 基本超参数

| 参数 | 取值 | 说明 |
|:---|:---:|:---|
| 训练轮数（Epochs） | 20 | 在验证集准确率趋于稳定时提前停止（实践中通过最优模型保存实现早停） |
| 批大小（Batch Size） | 128 | 平衡内存占用和梯度更新稳定性，每批处理 128 个样本 |
| 设备（Device） | `cuda`（若可用）或 `cpu` | 自动检测 GPU，利用 CUDA 加速训练 |

#### 2. 优化器（Optimizer）
- **类型**：`torch.optim.Adam`
- **学习率（lr）**：1e-3（0.001）
- **其他参数**：采用默认设置（`betas=(0.9, 0.999)`, `eps=1e-8`, `weight_decay=0`）
- **选择理由**：Adam 结合动量法和自适应学习率，对超参数不敏感，收敛快，适合中小规模模型。

#### 3. 损失函数（Loss Function）
- **类型**：`torch.nn.CrossEntropyLoss`
- **适用场景**：多分类任务（10 个类别）
- **说明**：该损失函数内部自动执行 **LogSoftmax** + **负对数似然（NLLLoss）**，因此模型输出层无需额外添加 Softmax，直接传入 logits 即可。

#### 4. 学习率调度器（Scheduler）
- **类型**：`torch.optim.lr_scheduler.ReduceLROnPlateau`
- **监控指标**：验证损失（`val_loss`）
- **模式**：`mode='min'`（损失下降视为改善）
- **触发条件**：连续 `patience=3` 个 epoch 验证损失未降低
- **调整方式**：学习率乘以 `factor=0.5`（减半）
- **作用**：在训练后期精细调整学习率，避免在局部最优附近震荡，提升最终性能。

#### 5. 梯度裁剪（Gradient Clipping）
- **方法**：`torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`
- **作用**：将梯度的 L2 范数截断至 1.0，防止梯度爆炸，尤其在深层网络或使用 Dropout 时稳定训练。

#### 6. 模型保存与早停（Early Stopping）
- **保存策略**：每个 epoch 结束后，若验证准确率（`val_acc`）超过当前最优值，则保存模型权重至 `best_model.pth`
- **早停**：虽然未显式设置早停轮数，但通过保存最优模型并在训练结束后加载该模型进行测试，实际上实现了“选择最佳验证点”，避免过拟合。

#### 7. 随机性控制（Reproducibility）
- **固定随机种子**：`np.random.seed(42)` 用于数据划分，确保每次划分的训练/验证集一致。
- **PyTorch 种子**（可选）：可增加 `torch.manual_seed(42)` 和 `torch.cuda.manual_seed_all(42)` 确保模型初始化一致，本实验未显式设置，但可通过添加实现完全复现。

#### 8. 多进程加载
- **`num_workers=2`**：使用 2 个子进程并行加载数据，提升 I/O 效率。在 Windows 下需注意主模块保护（`if __name__ == '__main__'`）以避免多进程冲突。

以上设置构成了一个完整、稳定的训练流程，适用于 MNIST 分类任务，同时具备良好的可移植性，可方便地迁移至其他数据集或模型。

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
num_epochs = 20
best_val_acc = 0
train_losses, val_losses, train_accs, val_accs = [], [], [], []
print(f"学习率：{1e-3}\nDevice:{device}")

学习率：0.001
Device:cuda


训练循环

In [ ]:
# ---------- 训练 / 评估函数 ----------
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)
    return total_loss / len(loader), correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        probs = torch.softmax(outputs, dim=1)

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return total_loss / len(loader), correct / total, all_preds, all_labels, all_probs

train_loader, val_loader, test_loader = get_mnist_loaders(batch_size=512, num_workers=0)


In [ ]:

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, _, _, _ = eval_epoch(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1:2d}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  → 保存最优模型 (val_acc = {best_val_acc:.4f})")

In [ ]:
# 加载最优模型并测试
model.load_state_dict(torch.load('best_model.pth'))
test_loss, test_acc, test_preds, test_labels, test_probs = \
    eval_epoch(model, test_loader, criterion, device)
print(f"\n最终测试结果：Loss={test_loss:.4f}, Accuracy={test_acc:.4f}")

# ---------- 新增：绘制训练曲线 ----------
save_path='training_curves.png'
   
epochs = range(1, len(train_losses) + 1)

%matplotlib inline

plt.figure(figsize=(12, 5))

# 子图1：损失曲线
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, 'b-', label='Train Loss')
plt.plot(epochs, val_losses, 'r-', label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curves')
plt.legend()
plt.grid(True)

# 子图2：准确率曲线
plt.subplot(1, 2, 2)
plt.plot(epochs, train_accs, 'b-', label='Train Acc')
plt.plot(epochs, val_accs, 'r-', label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Curves')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(save_path, dpi=300)
print(f"训练曲线已保存至 {save_path}")
plt.show()   # 若在非交互环境可注释掉